# Apresentação dos resultados - Processamento Paralelo

## A proposta

Este trabalho propõe o desenvolvimento e a análise de desempenho de um algoritmo para o cálculo de *Bounding Box* aplicado a conjuntos de dados multidimensionais. A implementação é realizada integralmente na linguagem C, focando em eficiência e controle de baixo nível sobre a memória.

### Objetivos Principais:
- **Estrutura de Dados Customizada:** Para gerenciar os dados de entrada, foi implementada uma estrutura chamada **Tensor**, que encapsula o ponteiro de dados e as informações de dimensionalidade, permitindo uma manipulação robusta de volumes de dados médicos ou científicos.
- **Implementação Sequencial:** Desenvolvimento de uma versão de referência executada em uma única thread, garantindo a corretude do cálculo dos limites espaciais da região de interesse.
- **Implementação Paralela com OpenMP:** Integração da biblioteca OpenMP para distribuir a carga de processamento entre os núcleos da CPU, visando reduzir o tempo de execução em grandes volumes de dados.
    - Os testes foram feitos com 2, 4, 8 e 16 threads.
    - Após testes, foi confirmado que o melhores números de threads era: 4 e 8;
    - Para os resutlados desse trabalho: formam utilizadas 4 threads.

### Estudo de Escalabilidade e Diretivas OpenMP:

O foco central é a comparação entre diferentes estratégias de escalonamento fornecidas pelo OpenMP. O objetivo é identificar qual abordagem de distribuição de iterações melhor se adapta à carga de trabalho do algoritmo de *Bounding Box*. Estão sendo comparadas as seguintes diretivas de loop for:
- **Sequencial:** Sem paralelismo.
- **Static:** Divisão de iterações em blocos fixos entre as threads.
- **Dynamic:** Distribuição de iterações conforme a disponibilidade das threads (ideal para cargas de trabalho desbalanceadas).
- **Guided:** Semelhante ao dinâmico, mas com blocos de tamanho decrescente para reduzir o overhead de sincronização.

Ao final desta análise, os resultados de tempo de execução e eficiência serão confrontados para determinar uma configuração de processamento paralelo para a estrutura de *Tensor* proposta.

## Implementação

### Implementação de funções utils

```c
#include <locale.h>
#include "../lib/config.h"

void configuration() {
    setlocale(LC_ALL, "Portuguese");
}
```

### Implementação do Tensor

```c
#include <stdio.h>
#include <stdlib.h>
#include <stddef.h>
#include <float.h>
#include "../lib/tensor.h"

void build_tensor_from_file(FILE *file, Ttensor *ptr_tensor) {
    size_t array_size;

    fscanf(file, "%lu", &ptr_tensor->x);
    fscanf(file, "%lu", &ptr_tensor->y);
    fscanf(file, "%lu", &ptr_tensor->z);
    array_size = ptr_tensor->x * ptr_tensor->y * ptr_tensor->z;
    allocate_array(ptr_tensor, array_size);

    for(size_t i = 0; i < array_size; i++) {
        fscanf(file, "%lf", (ptr_tensor->data + i));
    }

    return;
}

void destroy_tensor_data(Ttensor *ptr_tensor) {
    free(ptr_tensor->data);

    return;
}

void allocate_array(Ttensor *ptr_tensor, size_t data_size) {
    ptr_tensor->data = (double*) malloc(data_size * sizeof(double));
    
    if(ptr_tensor->data == NULL) {
        printf("Could not allocate enough space to save tensor data.\n");
        exit(-1);
    }

    return;
}
```

### Implementação sequencial

```c
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <stddef.h>
#include <float.h>
#include <stdint.h>
#include <string.h>
#include <time.h>
#include "../lib/config.h"
#include "../lib/tensor.h"

typedef struct coordinates {
    size_t x_begin;
    size_t x_end;
    size_t y_begin;
    size_t y_end;
    size_t z_begin;
    size_t z_end;
} Tcoordinates;

void binarize_data(Ttensor *ptr_tensor, const float THRESHOLD);
void check_pixel(const size_t i, const size_t j, const size_t k, const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates);
void get_indices_binary_search(const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates);
void crop_data(const Ttensor *ptr_input_tensor, Ttensor *ptr_output_tensor);

int main(const int argc, const char *argv[]) {
    FILE *input_file,
        *output_file;
    Ttensor tensor,
        croped_tensor;
    float threshold;
    const char *filename;
    clock_t start,
        global_start = clock();

    if(argc > 2) {
        filename = argv[1];
        threshold = atof(argv[2]);
    } else {
        filename = " ";
        threshold = 0.5;
    }

    configuration();
    input_file = fopen(filename, "r");

    if(input_file == NULL) {
        printf("Could not find file *%s*" endl, filename);
        exit(-1);
    }

    start = clock();
    printf("Reading data from input file..." endl);
    build_tensor_from_file(input_file, &tensor);
    printf("Read time: %lf" endl, (double) (clock() - start) / CLOCKS_PER_SEC);
    fclose(input_file);
    start = clock();
    printf("Running bounding box..." endl);
    binarize_data(&tensor, threshold);
    crop_data(&tensor, &croped_tensor);
    printf("Process time: %lf" endl, (double) (clock() - start) / CLOCKS_PER_SEC);
    output_file = fopen("output_sequential.txt", "w");

    if(output_file == NULL) {
        printf("Could not find output file." endl);
        exit(-1);
    }

    printf("Writing data to output file..." endl);
    start = clock();

    for(size_t k = 0; k < croped_tensor.z; k++) {
        for(size_t i = 0; i < croped_tensor.x; i++) {
            for(size_t j = 0; j < croped_tensor.y; j++) {
                size_t input_index = (i * croped_tensor.y * croped_tensor.z) + (j * croped_tensor.z) + k;

                fprintf(output_file, "%lf\t", croped_tensor.data[input_index]);
            }

            fprintf(output_file, endl);
        }

        fprintf(output_file, endl "---- ---- ----" endl endl);
    }

    printf("Write time: %lf" endl, (double) (clock() - start) / CLOCKS_PER_SEC);
    fclose(output_file);
    destroy_tensor_data(&tensor);
    destroy_tensor_data(&croped_tensor);
    printf("Total time: %lf" endl, (double) (clock() - global_start) / CLOCKS_PER_SEC);
    return 0;
}

void binarize_data(Ttensor *ptr_tensor, const float THRESHOLD) {
    for(size_t k = 0; k < ptr_tensor->z; k++) {
        for(size_t i = 0; i < ptr_tensor->x; i++) {
            for(size_t j = 0; j < ptr_tensor->y; j++) {
                size_t index = (i * ptr_tensor->y * ptr_tensor->z) + (j * ptr_tensor->z) + k;

                ptr_tensor->data[index] = (ptr_tensor->data[index] > THRESHOLD)? 1.0 : 0.0;
            }
        }
    }

    return;
}

void check_pixel(const size_t i, const size_t j, const size_t k, const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates) {
    size_t index = (i * ptr_tensor->y * ptr_tensor->z) + (j * ptr_tensor->z) + k;

    if(ptr_tensor->data[index]) {
        if(i < ptr_coordinates->x_begin) {
            ptr_coordinates->x_begin = i;
        }

        if(j < ptr_coordinates->y_begin) {
            ptr_coordinates->y_begin = j;
        }

        if(k < ptr_coordinates->z_begin) {
            ptr_coordinates->z_begin = k;
        }

        if(i > ptr_coordinates->x_end) {
            ptr_coordinates->x_end = i;
        }

        if(j > ptr_coordinates->y_end) {
            ptr_coordinates->y_end = j;
        }

        if(k > ptr_coordinates->z_end) {
            ptr_coordinates->z_end = k;
        }
    }

    return;
}

void get_indices_binary_search(const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates) {
    size_t X = ptr_tensor->x,
        Y = ptr_tensor->y,
        Z = ptr_tensor->z,
        mid_x = X / 2;

    ptr_coordinates->x_begin = X;
    ptr_coordinates->y_begin = Y;
    ptr_coordinates->z_begin = Z;
    ptr_coordinates->x_end = 0;
    ptr_coordinates->y_end = 0;
    ptr_coordinates->z_end = 0;

    for(int i = (int) mid_x; i >= 0; i--) {
        for(size_t j = 0; j < Y; j++) {
            for(size_t k = 0; k < Z; k++) {
                check_pixel((size_t) i, j, k, ptr_tensor, ptr_coordinates);
            }
        }
    }

    for(size_t i = mid_x + 1; i < X; i++) {
        for(size_t j = 0; j < Y; j++) {
            for(size_t k = 0; k < Z; k++) {
                check_pixel(i, j, k, ptr_tensor, ptr_coordinates);
            }
        }
    }

    return;
}

void crop_data(const Ttensor *ptr_input_tensor, Ttensor *ptr_output_tensor) {
    Tcoordinates coordinates;
    size_t slice_size = 0,
        output_index = 0;

    get_indices_binary_search(ptr_input_tensor, &coordinates);

    if(coordinates.x_end < coordinates.x_begin) {
        printf("Error: No data was found." endl);
        exit(-1);
    }

    ptr_output_tensor->x = coordinates.x_end - coordinates.x_begin + 1;
    ptr_output_tensor->y = coordinates.y_end - coordinates.y_begin + 1;
    ptr_output_tensor->z = coordinates.z_end - coordinates.z_begin + 1;
    allocate_array(ptr_output_tensor, ptr_output_tensor->x * ptr_output_tensor->y * ptr_output_tensor->z);
    slice_size = ptr_output_tensor->z * sizeof(double);

    for(size_t i = coordinates.x_begin; i <= coordinates.x_end; i++) {
        for(size_t j = coordinates.y_begin; j <= coordinates.y_end; j++) {
            size_t input_index = (i * ptr_input_tensor->y * ptr_input_tensor->z) + (j * ptr_input_tensor->z) + coordinates.z_begin;

            memcpy(&ptr_output_tensor->data[output_index], &ptr_input_tensor->data[input_index], slice_size);
            output_index += ptr_output_tensor->z;
        }
    }

    return;
}
```

### Implementação paralela com OpenMP

```c
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <stddef.h>
#include <string.h>
#include <strings.h>

#ifdef _OPENMP
    #include <omp.h>
#else
    #define omp_get_wtime() 0.0
#endif

#include "../lib/config.h"
#include "../lib/tensor.h"

typedef struct coordinates {
    size_t x_begin;
    size_t x_end;
    size_t y_begin;
    size_t y_end;
    size_t z_begin;
    size_t z_end;
} Tcoordinates;

void omp_configuration(const unsigned int number_of_threads);
void build_tensor_from_file(FILE *file, Ttensor *tensor);
void binarize_data(Ttensor *ptr_tensor, const float THRESHOLD);
void get_indices(const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates);
void crop_data(const Ttensor *ptr_input_tensor, Ttensor *ptr_output_tensor);
void print_load_balance(const long *thread_work, const int number_of_threads, const long total_work);

int main(const int argc, const char *argv[]) {
    FILE *input_file,
        *output_file;
    Ttensor tensor,
        croped_tensor;
    int number_of_threads;
    const char *filename;
    float threshold,
        start,
        global_start = omp_get_wtime();

    if(argc > 3) {
        filename = argv[1];
        threshold = atof(argv[2]);
        number_of_threads = atoi(argv[3]);
    } else {
        filename = " ";
        threshold = 0.5;
        number_of_threads = 4;
    }

    omp_set_num_threads(number_of_threads);
    configuration();
    input_file = fopen(filename, "r");

    if(input_file == NULL) {
        printf("Could not find file *%s*" endl, filename);
        exit(-1);
    }

    start = omp_get_wtime();
    printf("Reading data from input file..." endl);
    build_tensor_from_file(input_file, &tensor);
    printf("Read time: %f" endl, omp_get_wtime() - start);
    fclose(input_file);
    start = omp_get_wtime();
    printf("Running bounding box..." endl);
    binarize_data(&tensor, threshold);
    crop_data(&tensor, &croped_tensor);
    printf("Process time: %f" endl, omp_get_wtime() - start);
    output_file = fopen("output_openMP.txt", "w");

    if(output_file == NULL) {
        printf("Could not find output file." endl);
        exit(-1);
    }

    printf("Writing data to output file..." endl);
    start = omp_get_wtime();

    for(size_t k = 0; k < croped_tensor.z; k++) {
        for(size_t i = 0; i < croped_tensor.x; i++) {
            for(size_t j = 0; j < croped_tensor.y; j++) {
                size_t input_index = (i * croped_tensor.y * croped_tensor.z) + (j * croped_tensor.z) + k;

                fprintf(output_file, "%lf\t", croped_tensor.data[input_index]);
            }

            fprintf(output_file, endl);
        }

        fprintf(output_file, endl "---- ---- ----" endl endl);
    }

    printf("Write time: %f" endl, omp_get_wtime() - start);
    fclose(output_file);
    destroy_tensor_data(&tensor);
    destroy_tensor_data(&croped_tensor);
    printf("Total time: %f" endl, omp_get_wtime() - global_start);
    return 0;
}

void binarize_data(Ttensor *ptr_tensor, const float THRESHOLD) {
    size_t X = ptr_tensor->x,
        Y = ptr_tensor->y,
        Z = ptr_tensor->z;
    int number_of_threads = omp_get_max_threads();
    long *thread_work = (long*) calloc(number_of_threads, sizeof(long));

    #pragma omp parallel for schedule(guided, 4)
    for(size_t i = 0; i < X; i++) {
        for(size_t j = 0; j < Y; j++) {
            for(size_t k = 0; k < Z; k++) {
                size_t index = (i * Y * Z) + (j * Z) + k;
                
                ptr_tensor->data[index] = (ptr_tensor->data[index] > THRESHOLD)? 1.0 : 0.0;
                thread_work[omp_get_thread_num()]++;
            }
        }
    }

    printf("BINARIZE DATA" endl);
    print_load_balance(thread_work, number_of_threads, X * Y * Z);
    free(thread_work);
    return;
}

void get_indices(const Ttensor *ptr_tensor, Tcoordinates *ptr_coordinates) {
    size_t x_min = ptr_tensor->x,
        y_min = ptr_tensor->y,
        z_min = ptr_tensor->z,
        x_max = 0,
        y_max = 0,
        z_max = 0;
    int number_of_threads = omp_get_max_threads();
    long *thread_work = (long*) calloc(number_of_threads, sizeof(long));

    #pragma omp parallel for schedule(guided, 4) reduction(min:x_min, y_min, z_min) reduction(max:x_max, y_max, z_max)
    for(size_t k = 0; k < ptr_tensor->z; k++) {
        for(size_t i = 0; i < ptr_tensor->x; i++) {
            for(size_t j = 0; j < ptr_tensor->y; j++) {
                size_t index = (i * ptr_tensor->y * ptr_tensor->z) + (j * ptr_tensor->z) + k;

                if(ptr_tensor->data[index]) {
                    if(i < x_min) {
                        x_min = i;
                    }

                    if(j < y_min) {
                        y_min = j;
                    }

                    if(k < z_min) {
                        z_min = k;
                    }

                    if(i > x_max) {
                        x_max = i;
                    }

                    if(j > y_max) {
                        y_max = j;
                    }

                    if(k > z_max) {
                        z_max = k;
                    }
                }

                thread_work[omp_get_thread_num()]++;

            }
        }
    }

    ptr_coordinates->x_begin = x_min;
    ptr_coordinates->y_begin = y_min;
    ptr_coordinates->z_begin = z_min;
    ptr_coordinates->x_end = x_max;
    ptr_coordinates->y_end = y_max;
    ptr_coordinates->z_end = z_max;
    printf("GET INDICES" endl);
    print_load_balance(thread_work, number_of_threads, ptr_tensor->x * ptr_tensor->y * ptr_tensor->z);
    free(thread_work);
    return;
}

void crop_data(const Ttensor *ptr_input_tensor, Ttensor *ptr_output_tensor) {
    Tcoordinates coordinates;
    size_t total_elements,
        slice_size;
    int number_of_threads = omp_get_max_threads();
    long *thread_work = (long*) calloc(number_of_threads, sizeof(long));

    get_indices(ptr_input_tensor, &coordinates);

    if(coordinates.x_end < coordinates.x_begin) {
        printf("Error: No data was found.\n");
        exit(-1);
    }

    ptr_output_tensor->x = coordinates.x_end - coordinates.x_begin + 1;
    ptr_output_tensor->y = coordinates.y_end - coordinates.y_begin + 1;
    ptr_output_tensor->z = coordinates.z_end - coordinates.z_begin + 1;
    total_elements = ptr_output_tensor->x * ptr_output_tensor->y * ptr_output_tensor->z;
    allocate_array(ptr_output_tensor, total_elements);
    slice_size = ptr_output_tensor->z * sizeof(double);
    
    // collapse(1)
    #pragma omp parallel for schedule(guided, 4)
    for(size_t i = coordinates.x_begin; i <= coordinates.x_end; i++) {
        for(size_t j = coordinates.y_begin; j <= coordinates.y_end; j++) {
            size_t output_index = ((i - coordinates.x_begin) * ptr_output_tensor->y * ptr_output_tensor->z) + ((j - coordinates.y_begin) * ptr_output_tensor->z),
                input_index = (i * ptr_input_tensor->y * ptr_input_tensor->z) + (j * ptr_input_tensor->z) + coordinates.z_begin;

            memcpy(&ptr_output_tensor->data[output_index], &ptr_input_tensor->data[input_index], slice_size);
            thread_work[omp_get_thread_num()]++;
        }
    }

    printf("CROP DATA" endl);
    print_load_balance(thread_work, number_of_threads, ptr_input_tensor->x * ptr_input_tensor->y * ptr_input_tensor->z);
    free(thread_work);
    return;
}

void print_load_balance(const long *thread_work, const int number_of_threads, const long total_work) {
    long max_work = 0,
        min_work = total_work;
    double imbalance;

    for(int i = 0; i < number_of_threads; i++) {
        double percentage = (100.0 * thread_work[i]) / total_work;

        printf("Thread %d -> %ld iterations (%f)" endl, i, thread_work[i], percentage);

        if(thread_work[i] > max_work) {
            max_work = thread_work[i];
        }

        if(thread_work[i] < min_work) {
            min_work = thread_work[i];
        }
    }

    imbalance = ((double) (max_work - min_work) / max_work) * 100.0;

    printf("===== LOAD BALANCING ======" endl);
    printf("Max work: %ld" endl, max_work);
    printf("Min work: %ld" endl, min_work);
    printf("Imbalance: %f%%" endl, imbalance);
    printf("===========================" endl);
    return;
}


```

## Importando pacotes

In [1]:
import numpy as np
import pandas as pd

## Recuperando dataset

In [2]:
results_df: pd.DataFrame = pd.read_csv("data/results.csv")

## Testes

### Pré-processamento

In [3]:
results_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
0,1,9.736030,0.5,sequencial,NaN,NaN
1,2,9.643935,0.5,sequencial,NaN,NaN
2,3,9.603937,0.5,sequencial,NaN,NaN
3,4,9.587112,0.5,sequencial,NaN,NaN
4,5,9.576417,0.5,sequencial,NaN,NaN
5,6,5.820079,0.5,sequencial,NaN,NaN
6,7,5.733536,0.5,sequencial,NaN,NaN
7,1,9.588651,0.5,static,1.0,0.014493
8,2,9.406636,0.5,static,1.0,0.014493
9,3,10.894795,0.5,static,1.0,0.014493


In [4]:
results_df["nome_modo"].unique()

<StringArray>
['sequencial', 'static', 'dynamic', 'guided']
Length: 4, dtype: str

In [5]:
results_df["valor_modo"].unique()

array([nan,  1.,  2.,  4.,  8., 16.])

In [6]:
results_df["valor_modo"].isna().sum()

7

In [7]:
results_df["Fab"].isna().sum()

7

In [8]:
results_df["interacao"] = results_df["interacao"].astype(np.uint8)
results_df["tempo"] = results_df["tempo"].astype(np.float32)
results_df["threshold"] = results_df["threshold"].astype(np.float32)
results_df["nome_modo"] = results_df["nome_modo"].astype("category")
results_df["valor_modo"] = results_df["valor_modo"].fillna(0).astype(np.uint8)
results_df["valor_modo"] = results_df["valor_modo"].astype("category")
results_df["Fab"] = results_df["Fab"].astype(np.float64)
results_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
0,1,9.736030,0.5,sequencial,0,NaN
1,2,9.643935,0.5,sequencial,0,NaN
2,3,9.603937,0.5,sequencial,0,NaN
3,4,9.587112,0.5,sequencial,0,NaN
4,5,9.576417,0.5,sequencial,0,NaN
5,6,5.820079,0.5,sequencial,0,NaN
6,7,5.733536,0.5,sequencial,0,NaN
7,1,9.588651,0.5,static,1,0.014493
8,2,9.406636,0.5,static,1,0.014493
9,3,10.894795,0.5,static,1,0.014493


### Sequencial

In [9]:
sequencial_df: pd.DataFrame = results_df[results_df["nome_modo"] == "sequencial"]
sequencial_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
0,1,9.736030,0.5,sequencial,0,NaN
1,2,9.643935,0.5,sequencial,0,NaN
2,3,9.603937,0.5,sequencial,0,NaN
3,4,9.587112,0.5,sequencial,0,NaN
4,5,9.576417,0.5,sequencial,0,NaN
5,6,5.820079,0.5,sequencial,0,NaN
6,7,5.733536,0.5,sequencial,0,NaN


#### Tempo de execução

In [10]:
print("TEMPO")
print("Média:", sequencial_df["tempo"].mean())
print("Desvio padrão:", sequencial_df["tempo"].std())
print("Valor mínimo:", sequencial_df["tempo"].min())
print("Valor máximo:", sequencial_df["tempo"].max())

TEMPO
Média: 8.528721
Desvio padrão: 1.8808265
Valor mínimo: 5.733536
Valor máximo: 9.73603


### Static, 1

In [11]:
static_df: pd.DataFrame = results_df.query("nome_modo == 'static' and valor_modo == 1")
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
7,1,9.588651,0.5,static,1,0.014493
8,2,9.406636,0.5,static,1,0.014493
9,3,10.894795,0.5,static,1,0.014493
10,4,9.402572,0.5,static,1,0.014493
11,5,9.542489,0.5,static,1,0.014493
12,6,9.461038,0.5,static,1,0.014493
13,7,10.023134,0.5,static,1,0.014493


#### Tempo de execução

In [12]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 9.759902
Desvio padrão: 0.54400843
Valor mínimo: 9.402572
Valor máximo: 10.894795


#### Fator de balanceamento de carga (Fab)

In [13]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.01449275
Desvio padrão: 0.0
Valor mínimo: 0.01449275
Valor máximo: 0.01449275


### Static, 2

In [14]:
static_df: pd.DataFrame = results_df.query("nome_modo == 'static' and valor_modo == 2")
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
14,1,5.575633,0.5,static,2,0.028571
15,2,5.594479,0.5,static,2,0.028571
16,3,5.543972,0.5,static,2,0.028571
17,4,5.559714,0.5,static,2,0.028571
18,5,6.524377,0.5,static,2,0.028571
19,6,5.578726,0.5,static,2,0.028571
20,7,5.564609,0.5,static,2,0.028571


#### Tempo de execução

In [15]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 5.7059298
Desvio padrão: 0.3612515
Valor mínimo: 5.543972
Valor máximo: 6.524377


#### Fator de balanceamento de carga (Fab)

In [16]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.028571430000000002
Desvio padrão: 3.7474310104154816e-18
Valor mínimo: 0.02857143
Valor máximo: 0.02857143


### Static, 4

In [17]:
static_df: pd.DataFrame = results_df.query("nome_modo == 'static' and valor_modo == 4")
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
21,1,9.357284,0.5,static,4,0.042254
22,2,9.478352,0.5,static,4,0.042254
23,3,9.431457,0.5,static,4,0.042254
24,4,9.374154,0.5,static,4,0.042254
25,5,9.431161,0.5,static,4,0.042254
26,6,5.567645,0.5,static,4,0.042254
27,7,5.603625,0.5,static,4,0.042254


#### Tempo de execução

In [18]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 8.320525
Desvio padrão: 1.8687402
Valor mínimo: 5.567645
Valor máximo: 9.478352


#### Fator de balanceamento de carga (Fab)

In [19]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.042253519999999996
Desvio padrão: 7.494862020830963e-18
Valor mínimo: 0.04225352
Valor máximo: 0.04225352


### Static, 8

In [20]:
static_df: pd.DataFrame = results_df.query("nome_modo == 'static' and valor_modo == 8")
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
28,1,5.616897,0.5,static,8,0.111111
29,2,5.608164,0.5,static,8,0.111111
30,3,5.613281,0.5,static,8,0.111111
31,4,6.166797,0.5,static,8,0.111111
32,5,5.585134,0.5,static,8,0.111111
33,6,5.579253,0.5,static,8,0.111111
34,7,5.709627,0.5,static,8,0.111111


#### Tempo de execução

In [21]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 5.6970224
Desvio padrão: 0.21156731
Valor mínimo: 5.579253
Valor máximo: 6.166797


#### Fator de balanceamento de carga (Fab)

In [22]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.11111110999999999
Desvio padrão: 1.4989724041661926e-17
Valor mínimo: 0.11111111
Valor máximo: 0.11111111


### Static, 16

In [23]:
static_df: pd.DataFrame = results_df.query("nome_modo == 'static' and valor_modo == 16")
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
35,1,5.621770,0.5,static,16,0.2
36,2,5.625606,0.5,static,16,0.2
37,3,6.502947,0.5,static,16,0.2
38,4,5.645408,0.5,static,16,0.2
39,5,5.624359,0.5,static,16,0.2
40,6,5.645139,0.5,static,16,0.2
41,7,6.461827,0.5,static,16,0.2


#### Tempo de execução

In [24]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 5.8752937
Desvio padrão: 0.4150049
Valor mínimo: 5.62177
Valor máximo: 6.502947


#### Fator de balanceamento de carga (Fab)

In [25]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.19999999999999998
Desvio padrão: 2.997944808332385e-17
Valor mínimo: 0.2
Valor máximo: 0.2


### Static, todos juntos

In [26]:
static_df: pd.DataFrame = results_df[results_df["nome_modo"] == "static"]
static_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
7,1,9.588651,0.5,static,1,0.014493
8,2,9.406636,0.5,static,1,0.014493
9,3,10.894795,0.5,static,1,0.014493
10,4,9.402572,0.5,static,1,0.014493
11,5,9.542489,0.5,static,1,0.014493
12,6,9.461038,0.5,static,1,0.014493
13,7,10.023134,0.5,static,1,0.014493
14,1,5.575633,0.5,static,2,0.028571
15,2,5.594479,0.5,static,2,0.028571
16,3,5.543972,0.5,static,2,0.028571


#### Tempo de execução

In [27]:
print("TEMPO")
print("Média:", static_df["tempo"].mean())
print("Desvio padrão:", static_df["tempo"].std())
print("Valor mínimo:", static_df["tempo"].min())
print("Valor máximo:", static_df["tempo"].max())

TEMPO
Média: 7.0717344
Desvio padrão: 1.8990781
Valor mínimo: 5.543972
Valor máximo: 10.894795


#### Fator de balanceamento de carga (Fab)

In [28]:
print("Fab")
print("Média:", static_df["Fab"].mean())
print("Desvio padrão:", static_df["Fab"].std())
print("Valor mínimo:", static_df["Fab"].min())
print("Valor máximo:", static_df["Fab"].max())

Fab
Média: 0.07928576200000002
Desvio padrão: 0.0698914523872527
Valor mínimo: 0.01449275
Valor máximo: 0.2


### Dynamic, 1

In [29]:
dynamic_df: pd.DataFrame = results_df.query("nome_modo == 'dynamic' and valor_modo == 1")
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
42,1,9.415752,0.5,dynamic,1,0.291139
43,2,9.431909,0.5,dynamic,1,0.494624
44,3,9.429630,0.5,dynamic,1,0.269737
45,4,9.442116,0.5,dynamic,1,0.414365
46,5,9.369778,0.5,dynamic,1,0.470930
47,6,9.449298,0.5,dynamic,1,0.416667
48,7,9.420634,0.5,dynamic,1,0.471910


#### Tempo de execução

In [30]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 9.42273
Desvio padrão: 0.026046593
Valor mínimo: 9.369778
Valor máximo: 9.449298


#### Fator de balanceamento de carga (Fab)

In [31]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.4041959128571429
Desvio padrão: 0.08975506984669367
Valor mínimo: 0.26973684
Valor máximo: 0.49462366


### Dynamic, 2

In [32]:
dynamic_df: pd.DataFrame = results_df.query("nome_modo == 'dynamic' and valor_modo == 2")
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
49,1,5.496648,0.5,dynamic,2,0.441860
50,2,5.505139,0.5,dynamic,2,0.516854
51,3,5.485201,0.5,dynamic,2,0.370370
52,4,5.510289,0.5,dynamic,2,0.495049
53,5,5.508034,0.5,dynamic,2,0.510417
54,6,5.511183,0.5,dynamic,2,0.106061
55,7,5.550842,0.5,dynamic,2,0.439024


#### Tempo de execução

In [33]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 5.5096197
Desvio padrão: 0.020368658
Valor mínimo: 5.485201
Valor máximo: 5.550842


#### Fator de balanceamento de carga (Fab)

In [34]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.4113765628571429
Desvio padrão: 0.14406514040211588
Valor mínimo: 0.10606061
Valor máximo: 0.51685393


### Dynamic, 4

In [35]:
dynamic_df: pd.DataFrame = results_df.query("nome_modo == 'dynamic' and valor_modo == 4")
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
56,1,9.373476,0.5,dynamic,4,0.418605
57,2,9.425068,0.5,dynamic,4,0.458333
58,3,9.420016,0.5,dynamic,4,0.307692
59,4,9.474835,0.5,dynamic,4,0.512821
60,5,9.431231,0.5,dynamic,4,0.500000
61,6,5.512187,0.5,dynamic,4,0.441860
62,7,6.625755,0.5,dynamic,4,0.487805


#### Tempo de execução

In [36]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 8.466082
Desvio padrão: 1.6690518
Valor mínimo: 5.512187
Valor máximo: 9.474835


#### Fator de balanceamento de carga (Fab)

In [37]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.44673087857142857
Desvio padrão: 0.0697300249968162
Valor mínimo: 0.30769231
Valor máximo: 0.51282051


### Dynamic, 8

In [38]:
dynamic_df: pd.DataFrame = results_df.query("nome_modo == 'dynamic' and valor_modo == 8")
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
63,1,5.573108,0.5,dynamic,8,0.500000
64,2,5.605294,0.5,dynamic,8,0.521739
65,3,5.556590,0.5,dynamic,8,0.434783
66,4,5.586041,0.5,dynamic,8,0.222222
67,5,5.571325,0.5,dynamic,8,0.520000
68,6,5.557799,0.5,dynamic,8,0.428571
69,7,5.561795,0.5,dynamic,8,0.454545


#### Tempo de execução

In [39]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 5.573136
Desvio padrão: 0.017519414
Valor mínimo: 5.55659
Valor máximo: 5.605294


#### Fator de balanceamento de carga (Fab)

In [40]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.44026583428571425
Desvio padrão: 0.10366400504046118
Valor mínimo: 0.22222222
Valor máximo: 0.52173913


### Dynamic, 16

In [41]:
dynamic_df: pd.DataFrame = results_df.query("nome_modo == 'dynamic' and valor_modo == 16")
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
70,1,5.567066,0.5,dynamic,16,0.454545
71,2,5.576582,0.5,dynamic,16,0.400000
72,3,5.590791,0.5,dynamic,16,0.222222
73,4,5.528856,0.5,dynamic,16,0.300000
74,5,5.581992,0.5,dynamic,16,0.454545
75,6,5.558937,0.5,dynamic,16,0.400000
76,7,5.544265,0.5,dynamic,16,0.300000


#### Tempo de execução

In [42]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 5.56407
Desvio padrão: 0.021830356
Valor mínimo: 5.528856
Valor máximo: 5.590791


#### Fator de balanceamento de carga (Fab)

In [43]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.36161615999999996
Desvio padrão: 0.08873379390167932
Valor mínimo: 0.22222222
Valor máximo: 0.45454545


### Static, todos juntos

In [44]:
dynamic_df: pd.DataFrame = results_df[results_df["nome_modo"] == "dynamic"]
dynamic_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
42,1,9.415752,0.5,dynamic,1,0.291139
43,2,9.431909,0.5,dynamic,1,0.494624
44,3,9.429630,0.5,dynamic,1,0.269737
45,4,9.442116,0.5,dynamic,1,0.414365
46,5,9.369778,0.5,dynamic,1,0.470930
47,6,9.449298,0.5,dynamic,1,0.416667
48,7,9.420634,0.5,dynamic,1,0.471910
49,1,5.496648,0.5,dynamic,2,0.441860
50,2,5.505139,0.5,dynamic,2,0.516854
51,3,5.485201,0.5,dynamic,2,0.370370


#### Tempo de execução

In [45]:
print("TEMPO")
print("Média:", dynamic_df["tempo"].mean())
print("Desvio padrão:", dynamic_df["tempo"].std())
print("Valor mínimo:", dynamic_df["tempo"].min())
print("Valor máximo:", dynamic_df["tempo"].max())

TEMPO
Média: 6.9071274
Desvio padrão: 1.8533785
Valor mínimo: 5.485201
Valor máximo: 9.474835


#### Fator de balanceamento de carga (Fab)

In [46]:
print("Fab")
print("Média:", dynamic_df["Fab"].mean())
print("Desvio padrão:", dynamic_df["Fab"].std())
print("Valor mínimo:", dynamic_df["Fab"].min())
print("Valor máximo:", dynamic_df["Fab"].max())

Fab
Média: 0.4128370697142858
Desvio padrão: 0.10087384902456123
Valor mínimo: 0.10606061
Valor máximo: 0.52173913


### Guided, 1

In [47]:
guided_df: pd.DataFrame = results_df.query("nome_modo == 'guided' and valor_modo == 1").copy()
guided_df.head()

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
77,1,9.396616,0.5,guided,1,0.060606
78,2,9.536132,0.5,guided,1,0.372781
79,3,9.445127,0.5,guided,1,0.409938
80,4,11.499469,0.5,guided,1,0.371257
81,5,9.469010,0.5,guided,1,0.491429


#### Tempo de execução

In [48]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.64106
Desvio padrão: 2.2260342
Valor mínimo: 5.55377
Valor máximo: 11.499469


#### Fator de balanceamento de carga (Fab)

In [49]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.3238496042857143
Desvio padrão: 0.15867698675556105
Valor mínimo: 0.06060606
Valor máximo: 0.49142857


### Guided, 2

In [50]:
guided_df: pd.DataFrame = results_df.query("nome_modo == 'guided' and valor_modo == 2").copy()
guided_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
84,1,9.389120,0.5,guided,2,0.190141
85,2,9.451968,0.5,guided,2,0.417647
86,3,9.461125,0.5,guided,2,0.347561
87,4,9.496612,0.5,guided,2,0.497110
88,5,9.531858,0.5,guided,2,0.453488
89,6,5.556785,0.5,guided,2,0.500000
90,7,6.469254,0.5,guided,2,0.522222


#### Tempo de execução

In [51]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.479531
Desvio padrão: 1.7059708
Valor mínimo: 5.556785
Valor máximo: 9.531858


#### Fator de balanceamento de carga (Fab)

In [52]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.4183099014285715
Desvio padrão: 0.11689379853901052
Valor mínimo: 0.19014085
Valor máximo: 0.52222222


### Guided, 4

In [53]:
guided_df: pd.DataFrame = results_df.query("nome_modo == 'guided' and valor_modo == 4").copy()
guided_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
91,1,9.481663,0.5,guided,4,0.463415
92,2,9.484815,0.5,guided,4,0.341772
93,3,9.494643,0.5,guided,4,0.263889
94,4,9.530169,0.5,guided,4,0.263889
95,5,9.440854,0.5,guided,4,0.386667
96,6,5.547305,0.5,guided,4,0.342466
97,7,5.564983,0.5,guided,4,0.426966


#### Tempo de execução

In [54]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.36349
Desvio padrão: 1.9179667
Valor mínimo: 5.547305
Valor máximo: 9.530169


#### Fator de balanceamento de carga (Fab)

In [55]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.3555804671428572
Desvio padrão: 0.07623193825218705
Valor mínimo: 0.26388889
Valor máximo: 0.46341463


### Guided, 8

In [56]:
guided_df: pd.DataFrame = results_df.query("nome_modo == 'guided' and valor_modo == 8").copy()
guided_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
98,1,11.572992,0.5,guided,8,0.424242
99,2,9.555800,0.5,guided,8,0.490446
100,3,9.537188,0.5,guided,8,0.464481
101,4,9.504136,0.5,guided,8,0.215827
102,5,9.516364,0.5,guided,8,0.413580
103,6,5.588932,0.5,guided,8,0.385093
104,7,5.659814,0.5,guided,8,0.469945


#### Tempo de execução

In [57]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.705032
Desvio padrão: 2.2331498
Valor mínimo: 5.588932
Valor máximo: 11.572992


#### Fator de balanceamento de carga (Fab)

In [58]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.4090878957142857
Desvio padrão: 0.09270595182499825
Valor mínimo: 0.21582734
Valor máximo: 0.49044586


### Guided, 16

In [59]:
guided_df: pd.DataFrame = results_df.query("nome_modo == 'guided' and valor_modo == 16").copy()
guided_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
105,1,9.444173,0.5,guided,16,0.360000
106,2,9.515241,0.5,guided,16,0.204225
107,3,9.439447,0.5,guided,16,0.380952
108,4,9.461076,0.5,guided,16,0.250000
109,5,9.482745,0.5,guided,16,0.401361
110,6,6.765128,0.5,guided,16,0.269737
111,7,6.665881,0.5,guided,16,0.344156


#### Tempo de execução

In [60]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.681956
Desvio padrão: 1.3438879
Valor mínimo: 6.665881
Valor máximo: 9.515241


#### Fator de balanceamento de carga (Fab)

In [61]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.31577585
Desvio padrão: 0.07441268421905882
Valor mínimo: 0.20422535
Valor máximo: 0.40136054


### Guided, todos juntos

In [62]:
guided_df: pd.DataFrame = results_df[results_df["nome_modo"] == "guided"]
guided_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
77,1,9.396616,0.5,guided,1,0.060606
78,2,9.536132,0.5,guided,1,0.372781
79,3,9.445127,0.5,guided,1,0.409938
80,4,11.499469,0.5,guided,1,0.371257
81,5,9.469010,0.5,guided,1,0.491429
82,6,5.587298,0.5,guided,1,0.418079
83,7,5.553770,0.5,guided,1,0.142857
84,1,9.389120,0.5,guided,2,0.190141
85,2,9.451968,0.5,guided,2,0.417647
86,3,9.461125,0.5,guided,2,0.347561


#### Tempo de execução

In [63]:
print("TEMPO")
print("Média:", guided_df["tempo"].mean())
print("Desvio padrão:", guided_df["tempo"].std())
print("Valor mínimo:", guided_df["tempo"].min())
print("Valor máximo:", guided_df["tempo"].max())

TEMPO
Média: 8.574214
Desvio padrão: 1.8038262
Valor mínimo: 5.547305
Valor máximo: 11.572992


#### Fator de balanceamento de carga (Fab)

In [64]:
print("Fab")
print("Média:", guided_df["Fab"].mean())
print("Desvio padrão:", guided_df["Fab"].std())
print("Valor mínimo:", guided_df["Fab"].min())
print("Valor máximo:", guided_df["Fab"].max())

Fab
Média: 0.3645207437142857
Desvio padrão: 0.11056545169688128
Valor mínimo: 0.06060606
Valor máximo: 0.52222222


## Analisando os resultados

In [65]:
Fab_df: pd.DataFrame = results_df[results_df["nome_modo"] != "sequencial"]
Fab_df.head(10)

,interacao,tempo,threshold,nome_modo,valor_modo,Fab
7,1,9.588651,0.5,static,1,0.014493
8,2,9.406636,0.5,static,1,0.014493
9,3,10.894795,0.5,static,1,0.014493
10,4,9.402572,0.5,static,1,0.014493
11,5,9.542489,0.5,static,1,0.014493
12,6,9.461038,0.5,static,1,0.014493
13,7,10.023134,0.5,static,1,0.014493
14,1,5.575633,0.5,static,2,0.028571
15,2,5.594479,0.5,static,2,0.028571
16,3,5.543972,0.5,static,2,0.028571


#### Tempo de execução

In [66]:
print("TEMPO")
print("Média:", Fab_df["tempo"].mean())
print("Desvio padrão:", Fab_df["tempo"].std())
print("Valor mínimo:", Fab_df["tempo"].min())
print("Valor máximo:", Fab_df["tempo"].max())

TEMPO
Média: 7.517692
Desvio padrão: 1.983385
Valor mínimo: 5.485201
Valor máximo: 11.572992


#### Fator de balanceamento de carga (Fab)

In [67]:
print("Fab")
print("Média:", Fab_df["Fab"].mean())
print("Desvio padrão:", Fab_df["Fab"].std())
print("Valor mínimo:", Fab_df["Fab"].min())
print("Valor máximo:", Fab_df["Fab"].max())

Fab
Média: 0.28554785847619046
Desvio padrão: 0.17546937689944805
Valor mínimo: 0.01449275
Valor máximo: 0.52222222


### Melhor resultado (Fab)
A combinação que demonstrou o melhor resultado de balanceamento de carga foi `(static, 1)` com `0,01449275` **(1,45%)**.
<br>
Resultado de todas as iterações.
<br>
### Pior resultado (Fab)
A combinação que demonstrou o melhor resultado de balanceamento de carga foi `(dynamic, 4)` com `0,52222222` **(52,22%)**.
<br>
Resultado da iteração 4.
<br>

### Melhor resultado (Tempo de execução)
A combinação que demonstrou o menor tempo de execução foi `(dynamic, 2)` com `5,485201` segundos.
<br>
Resultado da iteração 3.
<br>
O melhor tempo do algoritmo iterativo foi `5,733536` segundos.
<br>
Resultado da iteração 7.
<br>
### Pior resultado (Tempo de execução)
A combinação que demonstrou o maior tempo de execução foi `(guided, 8)` com `11,572992` segundos.
<br>
Resultado da iteração 1.
<br>
O pior tempo do algoritmo iterativo foi `9,73603` segundos.
<br>
Resultado da iteração 1.
<br>